# UK Biobank Research Analysis Platform - Basic data extraction

(_Disclaimer: This notebook is delivered 'As-Is'. Notwithstanding anything to the contrary, DNAnexus will have no warranty, support, liability or other obligations with respect to Materials provided hereunder. The [MIT License](https://github.com/dnanexus/OpenBio/blob/master/LICENSE.md) applies to this notebook._)

### Prologue

This prologue has been created for consistency across notebooks, and to improve notebook portability across projects.

In [1]:
# Import packages
import pyspark
import dxpy
import dxdata

In [3]:
# Spark initialization (Done only once; do not rerun this cell unless you select Kernel -> Restart kernel).
sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)

In [4]:
# Automatically discover dispensed database name and dataset id
dispensed_database = dxpy.find_one_data_object(
    classname='database', 
    name='app*', 
    folder='/', 
    name_mode='glob', 
    describe=True)
dispensed_database_name = dispensed_database['describe']['name']

dispensed_dataset = dxpy.find_one_data_object(
    typename='Dataset', 
    name='app*.dataset', 
    folder='/', 
    name_mode='glob')
dispensed_dataset_id = dispensed_dataset['id']

## Access dataset

In [5]:
dataset = dxdata.load_dataset(id=dispensed_dataset_id)

### Dataset 'entities' are virtual tables linked to one another.

The main entity is 'participant' and corresponds to most pheno fields. Additional entities correspond to linked health care data.
Entities starting with 'hesin' are for hospital records; entities starting with 'gp' are for GP records, etc.

### Accessing the main 'participant' entity

In [7]:
participant = dataset['participant']

#### Looking up fields, given UKB showcase field id

If you know the field id but you are not sure if it is instanced or arrayed, and want to grab all instances/arrays (if any), use these:

In [9]:
# Returns all field objects for a given UKB showcase field id

def fields_for_id(field_id):
    from distutils.version import LooseVersion
    field_id = str(field_id)
    fields = participant.find_fields(name_regex=r'^p{}(_i\d+)?(_a\d+)?$'.format(field_id))
    return sorted(fields, key=lambda f: LooseVersion(f.name))

# Returns all field names for a given UKB showcase field id

def field_names_for_id(field_id):
    return [f.name for f in fields_for_id(field_id)]

def field_title_for_id(field_id):
    return [f.title for f in fields_for_id(field_id)]

In [14]:
# Returns all field objects for a given title keyword

def fields_by_title_keyword(keyword):
    from distutils.version import LooseVersion
    fields = list(participant.find_fields(lambda f: keyword.lower() in f.title.lower()))
    return sorted(fields, key=lambda f: LooseVersion(f.name))

# Returns all field names for a given title keyword

def field_names_by_title_keyword(keyword):
    return [f.name for f in fields_by_title_keyword(keyword)]

# Returns all field titles for a given title keyword

def field_titles_by_title_keyword(keyword):
    return [f.title for f in fields_by_title_keyword(keyword)]

In [15]:
field_titles_by_title_keyword('standing height')

['Standing height | Instance 0',
 'Standing height | Instance 1',
 'Standing height | Instance 2',
 'Standing height | Instance 3',
 'Reason for skipping standing height | Instance 0']

In [16]:
field_names_by_title_keyword('standing height')

['p50_i0', 'p50_i1', 'p50_i2', 'p50_i3', 'p20047_i0']

#### You can mix and match these methods to end up with a list of field names of interest:

In [32]:
field_names = ['eid', 'p31', 'p21022','p34','p40000_i0','p40000_i1','p40001_i0','p40001_i1'] \
+ field_names_for_id('131012') \
+ field_names_for_id('131013') \
+ field_names_for_id('131014') \
+ field_names_for_id('131015') \
+ field_names_for_id('131022') \
+ field_names_for_id('131023') \
+ field_names_for_id('131024') \
+ field_names_for_id('131025') \
+ field_names_for_id('131026') \
+ field_names_for_id('131027') \
+ field_names_for_id('131028') \
+ field_names_for_id('131029') \
+ field_names_for_id('131030') \
+ field_names_for_id('131031') \
+ field_names_for_id('131032') \
+ field_names_for_id('131033') \
+ field_names_for_id('131034') \
+ field_names_for_id('131035') \
+ field_names_for_id('131036') \
+ field_names_for_id('131037') \
+ field_names_for_id('131038') \
+ field_names_for_id('131039') \
+ field_names_for_id('131040') \
+ field_names_for_id('131041') \
+ field_names_for_id('130836') \
+ field_names_for_id('130837') \
+ field_names_for_id('130838') \
+ field_names_for_id('130839') \
+ field_names_for_id('130840') \
+ field_names_for_id('130841') \
+ field_names_for_id('130842') \
+ field_names_for_id('130843') \
+ field_names_for_id('130844') \
+ field_names_for_id('130845') \
+ field_names_for_id('130846') \
+ field_names_for_id('130847') \
+ field_names_for_id('130874') \
+ field_names_for_id('130875') \
+ field_names_for_id('130878') \
+ field_names_for_id('130879') \
+ field_names_for_id('130880') \
+ field_names_for_id('130881') \
+ field_names_for_id('130882') \
+ field_names_for_id('130883') \
+ field_names_for_id('130884') \
+ field_names_for_id('130885') \
+ field_names_for_id('130886') \
+ field_names_for_id('130887') \
+ field_names_for_id('130888') \
+ field_names_for_id('130889') \
+ field_names_for_id('130890') \
+ field_names_for_id('130891') \
+ field_names_for_id('130892') \
+ field_names_for_id('130893')

In [38]:
for f in field_names:
    print(f)
    print(field_title_for_id(f.replace('p','')))

eid
[]
p31
['Sex']
p21022
['Age at recruitment']
p131012
["Date G10 first reported (huntington's disease)"]
p131013
["Source of report of G10 (huntington's disease)"]
p131014
['Date G11 first reported (hereditary ataxia)']
p131015
['Source of report of G11 (hereditary ataxia)']
p131022
["Date G20 first reported (parkinson's disease)"]
p131023
["Source of report of G20 (parkinson's disease)"]
p131024
['Date G21 first reported (secondary parkinsonism)']
p131025
['Source of report of G21 (secondary parkinsonism)']
p131026
['Date G22 first reported (parkinsonism in diseases classified elsewhere)']
p131027
['Source of report of G22 (parkinsonism in diseases classified elsewhere)']
p131028
['Date G23 first reported (other degenerative diseases of basal ganglia)']
p131029
['Source of report of G23 (other degenerative diseases of basal ganglia)']
p131030
['Date G24 first reported (dystonia)']
p131031
['Source of report of G24 (dystonia)']
p131032
['Date G25 first reported (other extrapyramidal

In [44]:
df = participant.retrieve_fields(names=field_names, engine=dxdata.connect())

In [53]:
# See the first five entries as a Pandas DataFrame:d
d=df.toPandas()

In [54]:
cols=d.columns
replacements={}
for col in cols[1:]:
    name=field_title_for_id(col.replace('p',''))[0].replace("'","").replace(" ","_").replace("-","").replace("(","").replace(")","").replace(".","").replace("__","_")
    replacements[col]=col+'_'+name

d=d.rename(columns=replacements)

In [55]:
d.to_csv('neurological_disorders_ukbb.csv',index=False)